## import libraries

In [52]:
import pandas as pd
from geotessera import GeoTessera
from pathlib import Path
import os

## read datasets

In [53]:
data_root = Path("_data/crop_country_exports")

In [54]:
def get_inventory(root_path):
    inventory = []

    for file_path in root_path.glob("*.csv"):
        parts = file_path.stem.split('_')
        
        if len(parts) > 3:
            crop = "_".join(parts[:2])
            country = parts[-2]
            year = int(parts[-1])

        else:  
            crop = parts[0]
            country = parts[-2]
            year = int(parts[-1])

        inventory.append({
            "crop": crop,
            "country": country,
            "year": year,
            "path": file_path
        })

    df = pd.DataFrame(inventory)

    if not df.empty:
        df = df.sort_values(by=["crop", 'year']).reset_index(drop=True)

    return df

In [55]:
available_datasets = get_inventory(data_root)
available_datasets

,crop,country,year,path
0,clover,de,2011,_data/crop_country_exports/clover_de_2011.csv
1,clover,de,2012,_data/crop_country_exports/clover_de_2012.csv
2,clover,be,2018,_data/crop_country_exports/clover_be_2018.csv
3,clover,iti,2019,_data/crop_country_exports/clover_iti_2019.csv
4,clover,be,2019,_data/crop_country_exports/clover_be_2019.csv
5,clover,de,2019,_data/crop_country_exports/clover_de_2019.csv
6,clover,sk,2019,_data/crop_country_exports/clover_sk_2019.csv
7,clover,pt,2019,_data/crop_country_exports/clover_pt_2019.csv
8,clover,sk,2020,_data/crop_country_exports/clover_sk_2020.csv
9,clover,sk,2021,_data/crop_country_exports/clover_sk_2021.csv


## filter dataframe

In [56]:
year = 2019
crop = 'orchards_fruits'

In [57]:
def filter_df(df, crop, year):
    return df[(df['crop'] == crop) & (df['year'] == year)]

In [58]:
filtered_dataset = filter_df(available_datasets, crop, year)
filtered_dataset

,crop,country,year,path
14,orchards_fruits,sk,2019,_data/crop_country_exports/orchards_fruits_sk_...
15,orchards_fruits,be,2019,_data/crop_country_exports/orchards_fruits_be_...
16,orchards_fruits,de,2019,_data/crop_country_exports/orchards_fruits_de_...
17,orchards_fruits,pt,2019,_data/crop_country_exports/orchards_fruits_pt_...


In [59]:
def create_dataset(df):
    country_dfs = {}

    for country, group in df.groupby('country'):
        country_dfs[country] = pd.concat([pd.read_csv(p) for p in group['path']], ignore_index=True)
    
    return country_dfs

In [60]:
dataset = create_dataset(filtered_dataset)
dataset.keys()

dict_keys(['be', 'de', 'pt', 'sk'])

In [61]:
dataset['be'].head(1)

,country_id,year,hcat4_code,hcat4_name,centroid,long,lat
0,be,2019,3330100000,orchards_fruits,POINT (3.0701699319321794 51.18659781106289),3.07017,51.186598


## retrieve embeddings

In [62]:
gt = GeoTessera(embeddings_dir="_data/embeddings_dir")

In [63]:
def emb_meta_to_df(embedding, metadata, long, lat):
    df_row = pd.DataFrame([metadata])
    df_row['embedding'] = [embedding.flatten()] 
    df_row['point_long'] = long
    df_row['point_lat'] = lat
    return df_row

# after 
# oi = embeddings2.flatten()
# oi.reshape(1, 128).shape

In [64]:
def process_dataframe_embeddings(gdf, crop_name, country_id, year, include_metadate=True):
    coords = list(zip(gdf['long'], gdf['lat']))
    
    embeddings, metadata_list = gt.sample_embeddings_at_points(coords, 
                                                               year=year, 
                                                               include_metadata=include_metadate)
    
    all_rows = []
    for i in range(len(metadata_list)):
        meta = metadata_list[i]
        meta['crop'] = crop_name
        meta['country_id'] = country_id
        meta['year'] = year
        long = gdf['long'][i]
        lat = gdf['lat'][i]
        
        all_rows.append(emb_meta_to_df(embeddings[i], meta, long, lat))

    if all_rows:
        return pd.concat(all_rows, ignore_index=True)
    return pd.DataFrame()

In [65]:
embeddings = []

for key in dataset.keys():
    country = dataset[key]['country_id'][0]
    gdf = dataset[key]
    
    print(f"Extracting embeddings for: {key}...")
    result_df = process_dataframe_embeddings(gdf, crop, country, year, include_metadate=True)
    print(f"Embeddings for {key} extracted")

    embeddings.append(result_df)

Extracting embeddings for: be...
Embeddings for be extracted
Extracting embeddings for: de...
Embeddings for de extracted
Extracting embeddings for: pt...
Embeddings for pt extracted
Extracting embeddings for: sk...
Embeddings for sk extracted


## export data

In [66]:
export_folder = "_data/embeddings_exports"

if not os.path.exists(export_folder):
    os.makedirs(export_folder)
    print(f"Created directory: {export_folder}")

for df in embeddings:
    country = df['country_id'].iloc[0]
    file_name = f"{crop}_{country}_{year}.parquet"
    full_path = os.path.join(export_folder, file_name)
    df.to_parquet(full_path, index=False, engine='pyarrow')
    print(f"Saved: {full_path}")

# To load it back later:
# df = pd.read_parquet('country_crop_embeddings.parquet')

Saved: _data/embeddings_exports/orchards_fruits_be_2019.parquet
Saved: _data/embeddings_exports/orchards_fruits_de_2019.parquet
Saved: _data/embeddings_exports/orchards_fruits_pt_2019.parquet
Saved: _data/embeddings_exports/orchards_fruits_sk_2019.parquet
